In [8]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Checkbox, Dropdown, Button, VBox, Output
from IPython.display import display
from scipy import signal

%matplotlib inline

print("Бібліотеки завантажено!")

Бібліотеки завантажено!


In [9]:
t = np.linspace(0, 5, 1000)
sampling_rate = 1000 / 5

print(f"Часова вісь: 0-5 секунд, {len(t)} точок")
print(f"Частота дискретизації: {sampling_rate} Гц")

Часова вісь: 0-5 секунд, 1000 точок
Частота дискретизації: 200.0 Гц


In [10]:
class HarmonicApp:
    def __init__(self):
        self.noise = None
        self.noise_params = None
        self.harmonic_params = None

        self.default_params = {
            'amplitude': 1.0,
            'frequency': 1.0,
            'phase': 0.0,
            'noise_mean': 0.0,
            'noise_std': 0.2,
            'show_noise': True,
            'filter_enabled': True,
            'filter_type': 'Low-pass',
            'filter_cutoff': 5.0
        }

        self.create_widgets()
        self.setup_interaction()

    def create_widgets(self):
        self.amplitude = FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Амплітуда:')
        self.frequency = FloatSlider(min=0.5, max=5.0, step=0.1, value=1.0, description='Частота (Гц):')
        self.phase = FloatSlider(min=0.0, max=2*np.pi, step=0.1, value=0.0, description='Фаза (рад):')
        self.noise_mean = FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='Шум (середнє):')
        self.noise_std = FloatSlider(min=0.0, max=1.0, step=0.01, value=0.2, description='Шум (σ):')
        self.show_noise = Checkbox(value=True, description='Показати шум')
        self.filter_enabled = Checkbox(value=True, description='Увімкнути фільтр')
        self.filter_type = Dropdown(options=['Low-pass', 'High-pass', 'Band-pass'], value='Low-pass', description='Тип фільтра:')
        self.filter_cutoff = FloatSlider(min=0.5, max=10.0, step=0.5, value=5.0, description='Частота зрізу (Гц):')
        self.reset_button = Button(description='Reset', button_style='danger')
        self.output = Output()

    def generate_noise(self, mean, std, params_hash):
        if self.noise is None or self.noise_params != params_hash:
            self.noise = np.random.normal(mean, std, len(t))
            self.noise_params = params_hash
        return self.noise

    def harmonic_with_noise(self, amplitude, frequency, phase, noise_mean, noise_std, show_noise):
        clean = amplitude * np.sin(2 * np.pi * frequency * t + phase)
        if show_noise:
            noise_hash = (noise_mean, noise_std)
            noise = self.generate_noise(noise_mean, noise_std, noise_hash)
            return clean + noise, clean
        return clean, clean

    def apply_filter(self, signal_data, filter_type, cutoff):
        nyquist = sampling_rate / 2
        normalized_cutoff = cutoff / nyquist

        if normalized_cutoff >= 1:
            normalized_cutoff = 0.99

        if filter_type == 'Low-pass':
            b, a = signal.butter(4, normalized_cutoff, btype='low')
        elif filter_type == 'High-pass':
            b, a = signal.butter(4, normalized_cutoff, btype='high')
        else:
            b, a = signal.butter(4, [normalized_cutoff/2, normalized_cutoff], btype='band')

        return signal.filtfilt(b, a, signal_data)

    def update_plot(self, amplitude, frequency, phase, noise_mean, noise_std,
                    show_noise, filter_enabled, filter_type, filter_cutoff):

        signal_noise, clean = self.harmonic_with_noise(amplitude, frequency, phase,
                                                        noise_mean, noise_std, show_noise)

        if filter_enabled and show_noise:
            filtered = self.apply_filter(signal_noise, filter_type, filter_cutoff)
            mse = np.mean((filtered - clean)**2)
        else:
            filtered = clean
            mse = 0

        with self.output:
            self.output.clear_output(wait=True)

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

            if show_noise:
                ax1.plot(t, signal_noise, 'b-', alpha=0.7, linewidth=1, label='Зашумлений сигнал')
                ax1.plot(t, clean, 'r--', alpha=0.7, linewidth=1.5, label='Чиста гармоніка')
                ax1.set_title('Оригінальний сигнал (з шумом)')
            else:
                ax1.plot(t, clean, 'g-', linewidth=2, label='Чиста гармоніка')
                ax1.set_title('Чиста гармоніка')

            ax1.set_xlabel('Час (с)')
            ax1.set_ylabel('Амплітуда')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
            ax1.set_ylim(-3.5, 3.5)

            if filter_enabled and show_noise:
                ax2.plot(t, filtered, 'purple', linewidth=2, label=f'Відфільтрований ({filter_type})')
                ax2.plot(t, clean, 'r--', alpha=0.5, linewidth=1.5, label='Чиста гармоніка')
                ax2.set_title(f'Відфільтрований сигнал (похибка: {mse:.6f})')
            elif show_noise:
                ax2.plot(t, signal_noise, 'orange', alpha=0.7, label='Зашумлений (фільтр вимкнено)')
                ax2.plot(t, clean, 'r--', alpha=0.5, label='Чиста гармоніка')
                ax2.set_title('Фільтр вимкнено')
            else:
                ax2.plot(t, clean, 'g-', linewidth=2, label='Чиста гармоніка')
                ax2.set_title('Немає шуму')

            ax2.set_xlabel('Час (с)')
            ax2.set_ylabel('Амплітуда')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
            ax2.set_ylim(-3.5, 3.5)

            plt.tight_layout()
            plt.show()

    def reset_params(self, b):
        self.amplitude.value = self.default_params['amplitude']
        self.frequency.value = self.default_params['frequency']
        self.phase.value = self.default_params['phase']
        self.noise_mean.value = self.default_params['noise_mean']
        self.noise_std.value = self.default_params['noise_std']
        self.show_noise.value = self.default_params['show_noise']
        self.filter_enabled.value = self.default_params['filter_enabled']
        self.filter_type.value = self.default_params['filter_type']
        self.filter_cutoff.value = self.default_params['filter_cutoff']
        self.noise = None
        self.noise_params = None

    def setup_interaction(self):
        self.reset_button.on_click(self.reset_params)

        controls = VBox([
            self.amplitude, self.frequency, self.phase,
            self.noise_mean, self.noise_std, self.show_noise,
            self.filter_enabled, self.filter_type, self.filter_cutoff,
            self.reset_button
        ])

        display(controls, self.output)

        widgets = [self.amplitude, self.frequency, self.phase,
                   self.noise_mean, self.noise_std, self.show_noise,
                   self.filter_enabled, self.filter_type, self.filter_cutoff]

        def observer(change):
            self.update_plot(self.amplitude.value, self.frequency.value, self.phase.value,
                             self.noise_mean.value, self.noise_std.value, self.show_noise.value,
                             self.filter_enabled.value, self.filter_type.value, self.filter_cutoff.value)

        for w in widgets:
            w.observe(observer, names='value')

        self.update_plot(**self.default_params)

In [11]:
app = HarmonicApp()
print("Програма запущена! Використовуйте слайдери для керування.")

Output()

Програма запущена! Використовуйте слайдери для керування.
